In [4]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [5]:
RESULT_COLUMNS

{'label': 'binder', 'peptide': 'peptide', 'id': 'source_sequence_name'}

## Read/data

In [7]:
DATA_PATH = "/home/denklewer/Data/hmm_umap/"
datasets = ['bw', 'bwv', 'v']
data_types = ['train', 'test']

def read_file(dataset='bw', data_type='test'):
    file_name = "{}{}_{}_umap.csv".format(DATA_PATH, dataset,data_type)
    df =  pd.read_csv(file_name,sep=";", index_col=0)
    df['peptide'] = df.index.values
    df['source_sequence_name'] = [f"seq{i}" for i in range(len(df))]
    return df

# # read total data
# #train 
# bw_train_umap = read_file("{}bw_train_umap.csv".format(DATA_PATH))
# bwv_train_umap = read_file("{}bwv_train_umap.csv".format(DATA_PATH))
# v_train_umap =  read_file("{}v_train_umap.csv".format(DATA_PATH))

# bw_test_umap =  read_file("{}bw_test_umap.csv".format(DATA_PATH))
# bwv_test_umap = read_file("{}bwv_test_umap.csv".format(DATA_PATH))
# v_test_umap = read_file("{}v_test_umap.csv".format(DATA_PATH))
sample_df = df = read_file(dataset=datasets[0], data_type=data_types[0]) # same df for each allele

ALLELES = list(set(sample_df['allele']))

per_dataset_per_data_type_per_allele_df = dict()


# per_allele_bw_train_df =dict()
# per_allele_bwv_train_df =dict()
# per_allele_v_train_df =dict()

# per_allele_bw_test_df =dict()
# per_allele_bwv_test_df =dict()
# per_allele_v_test_df =dict()


for dataset in datasets:
    per_dataset_per_data_type_per_allele_df[dataset] = dict()
    for data_type in data_types:
        per_dataset_per_data_type_per_allele_df[dataset][data_type] = dict()
        for allele_name in ALLELES:
            target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')

            df = read_file(dataset=dataset, data_type=data_type) # same df for each allele
            assert 'peptide' in df.columns
            assert 'length' in df.columns
            assert RESULT_COLUMNS['id'] in df.columns
            assert 'peptide' in df.columns
            assert 'length' in df.columns
            assert RESULT_COLUMNS['id'] in df.columns
            per_dataset_per_data_type_per_allele_df[dataset][data_type][allele_name] = df



ALLELES

['HLA-A*03:01', 'HLA-A*01:01']

In [8]:
print(per_dataset_per_data_type_per_allele_df.keys())
print(per_dataset_per_data_type_per_allele_df[datasets[0]].keys())
print(per_dataset_per_data_type_per_allele_df[datasets[0]][data_types[0]].keys())

dict_keys(['bw', 'bwv', 'v'])
dict_keys(['train', 'test'])
dict_keys(['HLA-A*03:01', 'HLA-A*01:01'])


In [9]:
per_dataset_per_data_type_per_allele_df[datasets[0]][data_types[1]][ALLELES[0]]

,0,1,allele,dataset,length,peptide,source_sequence_name
YQDLLQLQY,8.424863,10.807470,HLA-A*01:01,Test_db,9,YQDLLQLQY,seq0
RTLLGLILFV,7.639996,6.766083,HLA-A*03:01,Test_db,10,RTLLGLILFV,seq1
LTSSSHTLTY,10.759052,6.954618,HLA-A*01:01,Test_db,10,LTSSSHTLTY,seq2
LLFLNDMGK,8.166666,-0.725282,HLA-A*03:01,Test_db,9,LLFLNDMGK,seq3
RLKEEAAMK,9.279162,0.702158,HLA-A*03:01,Test_db,9,RLKEEAAMK,seq4
...,...,...,...,...,...,...,...
RVSGPSSLR,9.452952,6.164794,HLA-A*03:01,Test_db,9,RVSGPSSLR,seq3559
LVDPASHTLY,9.163220,10.004796,HLA-A*01:01,Test_db,10,LVDPASHTLY,seq3560
NLENAAELY,9.210168,10.445720,HLA-A*01:01,Test_db,9,NLENAAELY,seq3561
RSAKTLTPK,10.574842,0.301874,HLA-A*03:01,Test_db,9,RSAKTLTPK,seq3562


In [10]:
per_dataset_per_data_type_per_method_per_allele_predictions = dict()


## 0.1 Define TARGET_ALLELES (alleles than should be passed to algorithm)

In [11]:
TARGET_ALLELES = dict()

## HARDCODE alelle name which will be passed to algorithm
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    else:
        target_allele_name = allele_name
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA-A*03:01': 'HLA-A*03:01', 'HLA-A*01:01': 'HLA-A*01:01'}

# 1. NetMHC-3.4

## 1.0 Define methods

In [12]:
from mhctools import NetMHC3
METHOD = 'netMHC-3.4'

def splitByLengthAndFilter(fasta_dict, min_length = 8):
    clean_fasta_dict = {k: v for k, v in fasta_dict.items() if not pd.isnull(v)}  #delete nan elements
    unique_lengths = list(set(map(len, list(clean_fasta_dict.values()))))
    unique_lengths = list(filter(lambda x: x>=min_length, unique_lengths ))
    per_length_dict_of_dicts = {target_length :{k: v for k, v in clean_fasta_dict.items() if len(v) == target_length} for target_length in unique_lengths}
    return per_length_dict_of_dicts

def getPredictor(allele_name):
    alleles = [allele_name]
    predictor = NetMHC3(
        alleles=alleles,
        program_name=METHOD)
    return predictor

def getSupportedAlleles():
    supported_alleles = NetMHC3._determine_supported_alleles(METHOD,"-A")
    return supported_alleles
    
    
def getSamplePredictionDf():
    protein_sequences = {
      "1L2Y": "NLYIQWLKD",
      "1L3Y": "ECDTINCER"
    }
    predictor = getPredictor("A*02:01")
    df = getPrediction(predictor=predictor, fasta_dict=protein_sequences, length=9)
    return df

def getPrediction(predictor, fasta_dict, length):
    predictions = predictor.predict_subsequences(fasta_dict,peptide_lengths=[length]).to_dataframe()
    return predictions
    
    
    
def fillDummyPrediction(fasta_dict, allele_name, length):
    sample_df = pd.DataFrame()
    sample_df['source_sequence_name'] = fasta_dict.keys()
    sample_df['offset'] = 0
    sample_df['peptide'] = fasta_dict.values()
    sample_df['allele'] = allele_name
    sample_df['score'] = np.nan
    sample_df['affinity'] = np.nan
    sample_df['percentile_rank'] = np.nan
    sample_df['prediction_method_name'] = np.nan
    sample_df['prediction_method_name'] = METHOD
    sample_df['length'] = length

    return sample_df
    
    
def predictSequences(dict_of_dicts, allele_name):
    supported_alleles = getSupportedAlleles()
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])# normalize allele name here 
    print(f"Predict binders for  dataset {allele_name} using alleles parameter {norm_allele_name}")
    if norm_allele_name in supported_alleles:
        predictor = getPredictor(norm_allele_name)
        binding_predictions = [getPrediction(predictor, fasta_dict, length)
                               for length, fasta_dict in dict_of_dicts.items()]
    else:
        print("!!!Unsupported allele, create dummy prediction!!!")
        binding_predictions = [fillDummyPrediction(fasta_dict,allele_name, length) for length, fasta_dict in dict_of_dicts.items()]
                    
    lengths = [len(prediction) for prediction in binding_predictions] 
    print(f"total processed peptides after filtering -  {sum(lengths)}")
    return binding_predictions

def predict_with_netmhc(per_allele_df):
    # make fasta-style dicts
   
    per_allele_fasta = {
        allele_name : {
            ID : PEPTIDE 
            for ID, PEPTIDE in zip(per_allele_df[allele_name][RESULT_COLUMNS['id']].values, per_allele_df[allele_name]['peptide'].values) 
        } 
        for allele_name in ALLELES 
    }
        # split by length
    per_allele_per_length_dict = dict() 
    for allele_name in ALLELES:
        fasta_dict = per_allele_fasta[allele_name]
        per_length_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
        #print(f"allele {allele_name} fasta splited into {len(per_length_dict_of_dicts)} dicts")
        per_allele_per_length_dict[allele_name] = per_length_dict_of_dicts

    # predict
    per_allele_predictions = dict()
    for allele_name, per_length_dict in per_allele_per_length_dict.items():
        print(f"initial length {len(per_allele_df[allele_name])}")

        predictions = predictSequences(per_length_dict, allele_name)
        per_allele_predictions[allele_name] = predictions
    # join lengths together
    per_allele_total_predictions = dict()
    for allele_name, list_of_df in per_allele_predictions.items():
        df_result = pd.concat(list_of_df, ignore_index=True)
        per_allele_total_predictions[allele_name] = df_result
    return per_allele_total_predictions
    

## 1.1. Check supported alleles

In [13]:
supported_alleles = getSupportedAlleles()

for allele_name in ALLELES:
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])
    if norm_allele_name in supported_alleles:
        print(f"Allele {allele_name}: OK")
    else:
        print(f"Allele {allele_name}: is not supported!")
        


Allele HLA-A*03:01: OK
Allele HLA-A*01:01: OK


## 1.2. Predict

In [14]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length
0,1L3Y,0,ECDTINCER,HLA-A*02:01,0.053,28329.0,None,netMHC-3.4,9
1,1L2Y,0,NLYIQWLKD,HLA-A*02:01,0.056,27238.0,None,netMHC-3.4,9


In [15]:
%%time

for dataset in datasets:
    per_dataset_per_data_type_per_method_per_allele_predictions[dataset] = dict()
    for data_type in data_types:
        per_dataset_per_data_type_per_method_per_allele_predictions[dataset][data_type] = dict()
    
for dataset in datasets:
    for data_type in data_types:
        per_allele_predictions = predict_with_netmhc(per_dataset_per_data_type_per_allele_df[dataset][data_type])
        per_dataset_per_data_type_per_method_per_allele_predictions[dataset][data_type][METHOD] = per_allele_predictions


# print(f"\t Mutated:")
# per_allele_mt_predictions = predict_with_netmhc(per_allele_mt_df)
# per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions

# print(f"\t Wild:")
# per_allele_wt_predictions = predict_with_netmhc(per_allele_wt_df)
# per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions


initial length 13336
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  13336
initial length 13336
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  13336
initial length 3564
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  3564
initial length 3564
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  3564
initial length 13336
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  13336
initial length 13336
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  13336
initial length 3564
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed pep

In [16]:
per_dataset_per_data_type_per_method_per_allele_predictions['bw']['train'][METHOD][ALLELES[0]]

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length
0,seq8322,0,AAAAALQAK,HLA-A*03:01,0.379,827.0,None,netMHC-3.4,9
1,seq11968,0,AAAACTTMK,HLA-A*03:01,0.583,90.0,None,netMHC-3.4,9
2,seq166,0,AAAAGGAGY,HLA-A*03:01,0.181,7019.0,None,netMHC-3.4,9
3,seq9435,0,AAAALGRAP,HLA-A*03:01,0.074,22383.0,None,netMHC-3.4,9
4,seq8880,0,AAAERDVVY,HLA-A*03:01,0.123,13235.0,None,netMHC-3.4,9
...,...,...,...,...,...,...,...,...,...
13331,seq2340,0,YVPFGPATEE,HLA-A*03:01,0.061,25963.0,None,netMHC-3.4,10
13332,seq8455,0,YVYEHTLTGK,HLA-A*03:01,0.657,40.0,None,netMHC-3.4,10
13333,seq9772,0,YWDGEKETYV,HLA-A*03:01,0.066,24387.0,None,netMHC-3.4,10
13334,seq10069,0,YYDPNPQSTV,HLA-A*03:01,0.067,24302.0,None,netMHC-3.4,10


# 2. NetMHC-4.0

In [17]:
from mhctools import NetMHC4
METHOD = 'netMHC-4.0'

In [18]:
def getPredictor(allele_name):
    alleles = [allele_name]
    predictor = NetMHC4(
        alleles=alleles,
        program_name=METHOD)
    return predictor
def getSupportedAlleles():
    supported_alleles = NetMHC4._determine_supported_alleles(METHOD,"-listMHC")
    return supported_alleles

In [19]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length
0,1L3Y,0,ECDTINCER,HLA-A*02:01,0.026,37794.71,75.0,netMHC-4.0,9
1,1L2Y,0,NLYIQWLKD,HLA-A*02:01,0.092,18478.21,25.0,netMHC-4.0,9


## 2.1. Check alleles

In [20]:
supported_alleles = getSupportedAlleles()

for allele_name in ALLELES:
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])
    if norm_allele_name in supported_alleles:
        print(f"Allele {allele_name}: OK")
    else:
        print(f"Allele {allele_name}: is not supported!")

Allele HLA-A*03:01: OK
Allele HLA-A*01:01: OK


## 2.2. Predict

In [21]:
%%time

for dataset in datasets:
    for data_type in data_types:
        per_allele_predictions = predict_with_netmhc(per_dataset_per_data_type_per_allele_df[dataset][data_type])
        per_dataset_per_data_type_per_method_per_allele_predictions[dataset][data_type][METHOD] = per_allele_predictions

initial length 13336
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  13336
initial length 13336
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  13336
initial length 3564
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  3564
initial length 3564
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  3564
initial length 13336
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  13336
initial length 13336
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  13336
initial length 3564
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed pep

# 3. NetMHCPan-4.1

In [22]:
from mhctools import NetMHCpan41

In [23]:
METHOD = 'netMHCpan-4.1'

In [24]:
def getPredictor(alleles):
    """
    mode - "elution_score" or "binding_affinity"
    """
    predictor_ba = NetMHCpan41(
        alleles=alleles,
        program_name=METHOD,
        mode="binding_affinity"
    )
    
    predictor_el = NetMHCpan41(
        alleles=alleles,
        program_name=METHOD,
        mode="elution_score"
    )
    return predictor_ba, predictor_el

def getSupportedAlleles():
    supported_alleles = NetMHCpan41._determine_supported_alleles(METHOD,"-listMHC")
    return supported_alleles

def reorderColumns(df):
    df = df[['source_sequence_name', 'offset', 'peptide', 'length', 'allele', 'score_ba', 'score_el', 'affinity',
                      'percentile_rank_ba', 'percentile_rank_el', 'prediction_method_name' ]]
    return df

def getPrediction(predictor, fasta_dict, length):
    predictor_ba, predictor_el = predictor
    df_ba = predictor_ba.predict_subsequences(fasta_dict, peptide_lengths=[length]).to_dataframe()
    df_el = predictor_el.predict_subsequences(fasta_dict, peptide_lengths=[length]).to_dataframe()
    
    df_ba = df_ba.rename(columns={"percentile_rank": "percentile_rank_ba"})
    df_ba = df_ba.rename(columns={"score": "score_ba"})
    df_ba["percentile_rank_el"] = df_el["percentile_rank"]
    df_ba["score_el"] = df_el["score"]
    
    predictions = reorderColumns(df_ba)
    return predictions

def fillDummyPrediction(fasta_dict, allele_name, length):
    sample_df = pd.DataFrame()
    sample_df['source_sequence_name'] = fasta_dict.keys()
    sample_df['offset'] = 0
    sample_df['peptide'] = fasta_dict.values()
    sample_df['allele'] = allele_name
    sample_df['score_ba'] = np.nan
    sample_df['score_el'] = np.nan
    sample_df['affinity'] = np.nan
    sample_df['percentile_rank_el'] = np.nan
    sample_df['percentile_rank_ba'] = np.nan
    sample_df['prediction_method_name'] = METHOD
    sample_df['length'] = length
    return reorderColumns(sample_df)
    

In [25]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,length,allele,score_ba,score_el,affinity,percentile_rank_ba,percentile_rank_el,prediction_method_name
0,1L3Y,0,ECDTINCER,9,HLA-A*02:01,0.017415,0.000061,41413.08,77.963,48.2,netMHCpan-4.1
1,1L2Y,0,NLYIQWLKD,9,HLA-A*02:01,0.074361,0.000089,22364.03,29.463,43.0,netMHCpan-4.1


## 2.1. Check alleles

In [26]:
supported_alleles = getSupportedAlleles()

for allele_name in ALLELES:
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])
    if norm_allele_name in supported_alleles:
        print(f"Allele {allele_name}: OK")
    else:
        print(f"Allele {allele_name}: is not supported!")

Allele HLA-A*03:01: OK
Allele HLA-A*01:01: OK


## 2.2. Predict

In [27]:
%%time
for dataset in datasets:
    for data_type in data_types:
        per_allele_predictions = predict_with_netmhc(per_dataset_per_data_type_per_allele_df[dataset][data_type])
        per_dataset_per_data_type_per_method_per_allele_predictions[dataset][data_type][METHOD] = per_allele_predictions

initial length 13336
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  13336
initial length 13336
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  13336
initial length 3564
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  3564
initial length 3564
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  3564
initial length 13336
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  13336
initial length 13336
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
total processed peptides after filtering -  13336
initial length 3564
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed pep

# 4. NetMHCIIPan-4.0

In [28]:
from mhctools import NetMHCIIpan4
METHOD = 'netMHCIIpan-4.0'

In [29]:
def getPredictor(alleles):
    """
    mode - "elution_score" or "binding_affinity"
    """
    predictor_ba = NetMHCIIpan4(
        alleles=alleles,
        program_name=METHOD,
        mode="binding_affinity"
    )
    
    predictor_el = NetMHCIIpan4(
        alleles=alleles,
        program_name=METHOD,
        mode="elution_score"
    )
    return predictor_ba, predictor_el

def getSamplePredictionDf():
    protein_sequences = {
      "1L2Y": "NLYIQWLKD",
      "1L3Y": "ECDTINCER"
    }
    predictor = getPredictor("DRB1_0101")
    df = getPrediction(predictor=predictor, fasta_dict=protein_sequences, length=9)
    return df

def getSupportedAlleles():
    supported_alleles = NetMHCIIpan4._determine_supported_alleles(METHOD,"-list")
    return supported_alleles


    

In [30]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,length,allele,score_ba,score_el,affinity,percentile_rank_ba,percentile_rank_el,prediction_method_name
0,1L3Y,0,ECDTINCER,9,HLA-DRA1*01:01-DRB1*01:01,0.023452,0.000001,38794.46,95.0,95.0,netMHCIIpan-4.0
1,1L2Y,0,NLYIQWLKD,9,HLA-DRA1*01:01-DRB1*01:01,0.088652,0.000050,19160.06,95.0,95.0,netMHCIIpan-4.0


## 4.1. Check alleles

In [31]:
supported_alleles = getSupportedAlleles()

for allele_name in ALLELES:
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])
    if norm_allele_name in supported_alleles:
        print(f"Allele {allele_name}: OK")
    else:
        print(f"Allele {allele_name}: is not supported!")

Allele HLA-A*03:01: is not supported!
Allele HLA-A*01:01: is not supported!


## 4.2. Predict

In [32]:
%%time
for dataset in datasets:
    for data_type in data_types:
        per_allele_predictions = predict_with_netmhc(per_dataset_per_data_type_per_allele_df[dataset][data_type])
        per_dataset_per_data_type_per_method_per_allele_predictions[dataset][data_type][METHOD] = per_allele_predictions


initial length 13336
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  13336
initial length 13336
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  13336
initial length 3564
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  3564
initial length 3564
Predict binders for  dataset HLA-A*01:01 using alleles parameter HLA-A*01:01
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  3564
initial length 13336
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  13336
initial length

# 5. Join predictions

In [33]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df

def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
    

# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [34]:
def joinPredictionsToSource(per_method_per_allele_predictions, per_allele_source_df):
    per_allele_result = dict()
    methods = list(per_method_per_allele_predictions.keys())
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        df_source = per_allele_source_df[allele_name].copy()
        for method in methods:
            print(f"method {method}")
            df_method = getPredictionResult(per_method_per_allele_predictions, allele_name, method)
            df_source = df_source.merge(df_method, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
#         df_source = df_source.rename(columns={col:  col + "_wt" for col in df_source.columns if col not in leave_columns})
        per_allele_result[allele_name] = df_source
    return per_allele_result
                
                

              

In [35]:
%%time
per_dataset_per_data_type_result = dict()


for dataset in datasets:
    per_dataset_per_data_type_result[dataset] = dict()
    for data_type in data_types:
        per_allele_per_method_predictions = per_dataset_per_data_type_per_method_per_allele_predictions[dataset][data_type]
        per_allele_source_df = per_dataset_per_data_type_per_allele_df[dataset][data_type]
        
        per_allele_result = joinPredictionsToSource(per_allele_per_method_predictions, per_allele_source_df)
        
        per_dataset_per_data_type_result[dataset][data_type] = per_allele_result




allele HLA-A*03:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*01:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*03:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*01:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*03:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*01:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*03:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*01:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*03:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA-A*01:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele HLA

# 5. For now just write to files

In [37]:
METHOD = 'NetMHC'

TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [38]:
# for decoy_key in DECOY_PREFIXES:

for dataset in datasets:
    for data_type in data_types:
        for allele_name in ALLELES:
            total_df = per_dataset_per_data_type_result[dataset][data_type][allele_name]
            allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
            total_df.to_csv("{}{}_{}_{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD, data_type, dataset,allele_name),sep=";", index=True)

In [39]:
datasets

['bw', 'bwv', 'v']

In [41]:
per_dataset_per_data_type_result['bw']['test'][ALLELES[1]]

,0,1,allele,dataset,length,peptide,source_sequence_name,netMHC-3.4_score,netMHC-3.4_affinity,netMHC-3.4_percentile_rank,...,netMHCpan-4.1_score_ba,netMHCpan-4.1_score_el,netMHCpan-4.1_affinity,netMHCpan-4.1_percentile_rank_ba,netMHCpan-4.1_percentile_rank_el,netMHCIIpan-4.0_score_ba,netMHCIIpan-4.0_score_el,netMHCIIpan-4.0_affinity,netMHCIIpan-4.0_percentile_rank_ba,netMHCIIpan-4.0_percentile_rank_el
0,8.424863,10.807470,HLA-A*01:01,Test_db,9,YQDLLQLQY,seq0,0.747,15.0,None,...,0.637939,0.950884,50.27,0.048,0.031,NaN,NaN,NaN,NaN,NaN
1,7.639996,6.766083,HLA-A*03:01,Test_db,10,RTLLGLILFV,seq1,0.112,14943.0,None,...,0.111011,0.001037,15042.93,5.458,18.298,NaN,NaN,NaN,NaN,NaN
2,10.759052,6.954618,HLA-A*01:01,Test_db,10,LTSSSHTLTY,seq2,0.784,10.0,None,...,0.684041,0.883888,30.53,0.029,0.063,NaN,NaN,NaN,NaN,NaN
3,8.166666,-0.725282,HLA-A*03:01,Test_db,9,LLFLNDMGK,seq3,0.081,20719.0,None,...,0.069436,0.000615,23588.06,13.320,24.761,NaN,NaN,NaN,NaN,NaN
4,9.279162,0.702158,HLA-A*03:01,Test_db,9,RLKEEAAMK,seq4,0.075,22120.0,None,...,0.031312,0.001019,35631.65,48.707,18.489,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3559,9.452952,6.164794,HLA-A*03:01,Test_db,9,RVSGPSSLR,seq3559,0.078,21598.0,None,...,0.054626,0.010426,27687.52,20.760,4.768,NaN,NaN,NaN,NaN,NaN
3560,9.163220,10.004796,HLA-A*01:01,Test_db,10,LVDPASHTLY,seq3560,0.741,16.0,None,...,0.750919,0.989037,14.81,0.014,0.006,NaN,NaN,NaN,NaN,NaN
3561,9.210168,10.445720,HLA-A*01:01,Test_db,9,NLENAAELY,seq3561,0.556,122.0,None,...,0.545212,0.860250,137.10,0.109,0.074,NaN,NaN,NaN,NaN,NaN
3562,10.574842,0.301874,HLA-A*03:01,Test_db,9,RSAKTLTPK,seq3562,0.081,20788.0,None,...,0.087027,0.005652,19499.91,8.632,6.767,NaN,NaN,NaN,NaN,NaN


In [187]:
per_dataset_per_data_type_result['bw']['test'][ALLELES[0]]

,0,1,allele,dataset,length,peptide,source_sequence_name,netMHC-3.4_score,netMHC-3.4_affinity,netMHC-3.4_percentile_rank,...,netMHCpan-4.1_score_ba,netMHCpan-4.1_score_el,netMHCpan-4.1_affinity,netMHCpan-4.1_percentile_rank_ba,netMHCpan-4.1_percentile_rank_el,netMHCIIpan-4.0_score_ba,netMHCIIpan-4.0_score_el,netMHCIIpan-4.0_affinity,netMHCIIpan-4.0_percentile_rank_ba,netMHCIIpan-4.0_percentile_rank_el
0,11.584837,6.236691,HLA-A*01:01,Test_db,9,TSSSHTLTY,seq0,0.499,226.0,None,...,0.400527,0.369139,656.00,1.372,0.696,NaN,NaN,NaN,NaN,NaN
1,8.231989,6.037734,HLA-A*02:01,Test_db,9,TPNEERNVM,seq1,0.068,23932.0,None,...,0.027665,0.000302,37065.76,60.988,22.257,NaN,NaN,NaN,NaN,NaN
2,11.193851,7.000565,HLA-A*01:01,Test_db,9,PTAIYTLEY,seq2,0.128,12491.0,None,...,0.163092,0.006709,8562.60,6.564,6.531,NaN,NaN,NaN,NaN,NaN
3,3.737058,9.639633,HLA-A*01:01,Test_db,9,VADVVKDAY,seq3,0.071,23129.0,None,...,0.057500,0.001350,26839.79,27.415,12.377,NaN,NaN,NaN,NaN,NaN
4,11.123215,6.985098,HLA-A*03:01,Test_db,9,NFSIIELPY,seq4,0.091,18653.0,None,...,0.145240,0.002157,10387.03,7.658,10.346,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5324,5.569105,10.118204,HLA-A*01:01,Test_db,9,FVETESVRY,seq5324,0.088,19285.0,None,...,0.072266,0.002926,22876.75,20.160,9.169,NaN,NaN,NaN,NaN,NaN
5325,1.839018,5.130972,HLA-A*03:01,Test_db,9,SLNKQIAQK,seq5325,0.659,40.0,None,...,0.705600,0.963822,24.18,0.079,0.010,NaN,NaN,NaN,NaN,NaN
5326,10.864838,0.826170,HLA-A*03:01,Test_db,9,KLSFPLLAK,seq5326,0.742,16.0,None,...,0.822411,0.984547,6.83,0.010,0.004,NaN,NaN,NaN,NaN,NaN
5327,10.032868,1.185634,HLA-A*03:01,Test_db,9,RLRAEAQRK,seq5327,0.653,42.0,None,...,0.686995,0.874077,29.57,0.111,0.063,NaN,NaN,NaN,NaN,NaN
